DEEP LEARNING PROJECT: BINARY CLASSIFICATION OF NDVI SATELLITE IMAGES (FOREST/NOT FOREST)

In [1]:
#libs
import numpy as np
import rasterio
import os
from PIL import Image
import zarr

This script generates random points for labeling.

In [2]:
#Load the NDVI GeoTIFF image for training data extraction
image= r"C:\Users\mvens\Documents\projets\binary_classifier_forest\S2_NDVI_DrySeason_2024.tif"
with rasterio.open(image) as src:
    ndvi_array = src.read(1)

#Extract coordinates for 'forest' and 'non-forest' classes based on NDVI threshold
#Threshold >= 0.6 is considered forest, < 0.6 is non-forest
indices_forest = np.argwhere(ndvi_array >= 0.6)
indices_not_forest = np.argwhere(ndvi_array < 0.6)

In [3]:
#Patch and image configuration
size = 32
radius = size // 2
h, w = ndvi_array.shape # Dimensions image originale

#Filtering: Exclude pixels too close to the image boundaries 
#to ensure a full 32x32 patch can be extracted around each point
def edges_filter(indices, h, w, r):
    #Condition : y et x doivent etre compris entre [rayon] et [dimension - rayon]
    inside_mask = (indices[:, 0] >= r) & (indices[:, 0] < h - r) & \
                       (indices[:, 1] >= r) & (indices[:, 1] < w - r)
    return indices[inside_mask]

In [4]:
#Apply boundary filtering to ensure valid patch extraction
indices_forest_safe = edges_filter(indices_forest, h, w, radius)
indices_not_forest_safe = edges_filter(indices_not_forest, h, w, radius)

#Randomly sample 500 indices for each class to create a balanced dataset
#'replace=False' ensures unique points are selected
random_forest = np.random.choice(len(indices_forest_safe), 500, replace=False)
random_not_forest = np.random.choice(len(indices_not_forest_safe), 500, replace=False)

#Retrieve the (y, x) coordinates for the selected centroids
centroids_forest = indices_forest_safe[random_forest]
centroids_not_forest = indices_not_forest_safe[random_not_forest]

In [8]:
#Initialize Zarr storage for the training dataset
#Create datasets for each class with optimized chunking
# Shape: (Samples, Height, Width) | dtype: float32 ('f4')
output_zarr = "dataset_ndvi.zarr"

root = zarr.open(output_zarr, mode='w')
z_forest = root.create_dataset('forest', shape=(500, size, size), dtype='f4', chunks=(100, size, size))
z_not_forest = root.create_dataset('not_forest', shape=(500, size, size), dtype='f4', chunks=(100, size, size))

#Function to extract patches around centroids and stream them to Zarr
def extract_to_zarr(centroids, z_dataset):
    for i, (y, x) in enumerate(centroids):
        #Slice the NDVI array around the centroid (32x32 patch)
        patch = ndvi_array[y-radius : y+radius, x-radius : x+radius]
        
        #Save patch directly to the Zarr dataset at index i
        #Preserves float precision for better model training
        z_dataset[i] = patch

#Execute extraction for both classes
extract_to_zarr(centroids_forest, z_forest)
extract_to_zarr(centroids_not_forest, z_not_forest)

print(f"Dataset successfully saved to Zarr format at : {output_zarr}")

C:\Users\mvens\AppData\Local\Temp\ipykernel_25792\1697251999.py:7: ZarrDeprecationWarning: Use Group.create_array instead.
  z_forest = root.create_dataset('forest', shape=(500, size, size), dtype='f4', chunks=(100, size, size))
C:\Users\mvens\AppData\Local\Temp\ipykernel_25792\1697251999.py:8: ZarrDeprecationWarning: Use Group.create_array instead.
  z_not_forest = root.create_dataset('not_forest', shape=(500, size, size), dtype='f4', chunks=(100, size, size))


Dataset successfully saved to Zarr format at : dataset_ndvi.zarr
